# CardioIA — Fase 2, Parte 2
## Classificação de risco em texto clínico (TF-IDF + modelo supervisionado)

As frases em `frases_risco.csv` foram geradas a partir do dataset numérico da Fase 1 (`dataset_cardiologia.csv`): o rótulo **alto risco** corresponde a `historico_doenca_cardiaca > 0` e **baixo risco** a valor zero.

**Google Colab:** instale dependências se precisar (`!pip install pandas scikit-learn`) e aponte `DATA` para a pasta que contém `frases_risco.csv` (clone do GitHub ou upload).

In [ ]:
from pathlib import Path

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

candidatos = [
    Path("datasets"),
    Path("../datasets"),
    Path("/content/datasets"),
]
DATA = next((p for p in candidatos if (p / "frases_risco.csv").exists()), Path("datasets"))
print("Pasta de dados:", DATA.resolve())

In [ ]:
df = pd.read_csv(DATA / "frases_risco.csv", encoding="utf-8")
df["situacao"] = df["situacao"].str.strip().str.lower()
print(df["situacao"].value_counts())
df.head(3)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df["frase"],
    df["situacao"],
    test_size=0.25,
    random_state=42,
    stratify=df["situacao"],
)

vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_df=0.95)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

In [ ]:
modelo_lr = LogisticRegression(max_iter=2000, random_state=42)
modelo_lr.fit(X_train_tfidf, y_train)
pred_lr = modelo_lr.predict(X_test_tfidf)
print("=== Regressão logística ===")
print("Acurácia:", accuracy_score(y_test, pred_lr))
print(classification_report(y_test, pred_lr, digits=3))
print("Matriz de confusão:\n", confusion_matrix(y_test, pred_lr))

In [ ]:
modelo_arvore = DecisionTreeClassifier(max_depth=8, random_state=42)
modelo_arvore.fit(X_train_tfidf, y_train)
pred_arv = modelo_arvore.predict(X_test_tfidf)
print("=== Árvore de decisão ===")
print("Acurácia:", accuracy_score(y_test, pred_arv))
print(classification_report(y_test, pred_arv, digits=3))

In [ ]:
exemplos = [
    "sinto dor no peito intensa e falta de ar mesmo em repouso",
    "exames de rotina normais, sem dor nem falta de ar",
]
X_novo = vectorizer.transform(exemplos)
for texto, r in zip(exemplos, modelo_lr.predict(X_novo)):
    print(f"{r!r}: {texto}")

## Viés e governança (reflexão)

- Os rótulos vieram de uma **regra simples** sobre o CSV da Fase 1, não de laudo médico: o modelo aprende essa convenção, não “verdade clínica”.
- **Desbalanceamento** entre classes pode favorecer a classe majoritária na acurácia.
- Frases **muito parecidas** entre pacientes diferentes podem levar a **vazamento** de padrões espúrios; em triagem real seriam necessários auditoria, equidade entre grupos e explicabilidade.

Use esta seção no vídeo da entrega para comentar limitações éticas do protótipo.